In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118


In [11]:
import os
import numpy as np
import torch
from PIL import Image
from facenet_pytorch import MTCNN
import shutil


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mtcnn = MTCNN(
    image_size=160,
    margin=20,
    device=device
)

input_root = "resnet_train"
output_root = "resnet_train_aligned"

os.makedirs(output_root, exist_ok=True)

for img_name in os.listdir(input_root):
    img_path = os.path.join(input_root, img_name)
    image_name = img_name.split("_")[0]
    if not os.path.isdir(os.path.join(output_root, image_name)):
        os.mkdir(os.path.join(output_root, image_name))
    if not os.path.isfile(img_path):
        continue
    shutil.copy2(img_path, os.path.join(output_root, image_name)) 
    

In [12]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

data_dir = "resnet_train_aligned"

# Аугментации (преобразования) для тренировки – делаем данные более разнообразными
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),       # случайное отражение
    transforms.RandomRotation(10),           # небольшой поворот
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # яркость/контраст
    transforms.ToTensor(),                   # переводим в тензор [0,1]
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # нормализация
])

# Для валидации – без случайных искажений
val_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Загружаем все данные, потом разделим на train/val
full_dataset = datasets.ImageFolder(root=data_dir, transform=train_transforms)

# Разделим вручную: 80% тренировка, 20% тест
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

# Для validation dataset используем другую трансформацию
val_dataset.dataset.transform = val_transforms   # немного костыль, но для простоты сработает

# DataLoader'ы
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Сохраним количество классов (людей)
num_classes = len(full_dataset.classes)
print(f"Классов (людей): {num_classes}")
print(f"Имена классов: {full_dataset.classes}")

Классов (людей): 31
Имена классов: ['Akshay Kumar', 'Alexandra Daddario', 'Alia Bhatt', 'Amitabh Bachchan', 'Andy Samberg', 'Anushka Sharma', 'Billie Eilish', 'Brad Pitt', 'Camila Cabello', 'Charlize Theron', 'Claire Holt', 'Courtney Cox', 'Dwayne Johnson', 'Elizabeth Olsen', 'Ellen Degeneres', 'Henry Cavill', 'Hrithik Roshan', 'Hugh Jackman', 'Jessica Alba', 'Kashyap', 'Lisa Kudrow', 'Margot Robbie', 'Marmik', 'Natalie Portman', 'Priyanka Chopra', 'Robert Downey Jr', 'Roger Federer', 'Tom Cruise', 'Vijay Deverakonda', 'Virat Kohli', 'Zac Efron']


In [13]:
import torch.nn as nn
from torchvision import models

class FaceModel(nn.Module):
    def __init__(self, num_classes, embedding_dim=128):
        super().__init__()
        # Берем предобученный ResNet18
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # Убираем оригинальный fully connected слой
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()   # теперь backbone выдаёт вектор размером in_features (512)
        
        # Слой, который превратит признаки в компактный эмбеддинг (вектор лица)
        self.embedding = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Linear(256, embedding_dim)
        )
        
        # Классификатор (только для обучения)
        self.classifier = nn.Linear(embedding_dim, num_classes)
        
    def forward(self, x, return_embedding=False):
        features = self.backbone(x)          # (batch, 512)
        emb = self.embedding(features)       # (batch, embedding_dim)
        if return_embedding:
            return emb
        out = self.classifier(emb)           # (batch, num_classes)
        return out

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class ArcFaceModel(nn.Module):
    def __init__(self, num_classes, embedding_dim=512):
        super().__init__()
        # Backbone (ResNet18)
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        
        # Эмбеддинг: пара полносвязных слоёв, можно и один
        self.embedding = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, embedding_dim),
            nn.BatchNorm1d(embedding_dim)
        )
        
        # Классификатор без bias, веса будут нормализованы
        self.classifier = nn.Linear(embedding_dim, num_classes, bias=False)
        
        # Параметры ArcFace (можно сделать аргументами)
        self.s = 64.0   # scale
        self.m = 0.5    # margin в радианах (угловой отступ)
        
        # Инициализация весов классификатора (можно ксавьером)
        nn.init.xavier_normal_(self.classifier.weight)
    
    def forward(self, x, labels=None):
        # Извлекаем признаки и эмбеддинг
        features = self.backbone(x)
        emb = self.embedding(features)
        emb = F.normalize(emb, p=2, dim=1)   # L2 нормализация эмбеддинга
        
        # Если нам нужен только вектор лица (инференс)
        if labels is None:
            return emb
        
        # Нормализуем веса классификатора
        w = F.normalize(self.classifier.weight, p=2, dim=1)  # [num_classes, embedding_dim]
        
        # Вычисляем косинус угла: (emb * w^T) = cos(theta)
        cosine = torch.matmul(emb, w.t())  # [batch, num_classes]
        
        # Преобразуем в угол (арккосинус) и добавляем margin
        theta = torch.acos(cosine.clamp(-1.0, 1.0))   # угол в радианах
        # Добавляем margin только к правильному классу
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        theta_with_margin = theta + one_hot * self.m
        
        # Обратно в косинус и умножаем на scale
        logits = self.s * torch.cos(theta_with_margin)
        
        return logits, emb

In [19]:
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available(), device)
# Параметры датасета (как и раньше)
data_dir = "resnet_train_aligned"   # папка с подпапками людей
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

dataset = datasets.ImageFolder(root=data_dir, transform=train_transforms)
num_classes = len(dataset.classes)
print("Количество людей:", num_classes)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

model = ArcFaceModel(num_classes=num_classes, embedding_dim=512).to(device)
if os.path.isfile("arcface_model.pth"):
    print("Loaded")
    model.load_state_dict(torch.load("arcface_model.pth", map_location=device))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

epochs = 
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        logits, _ = model(images, labels)   # передаём метки для расчёта ArcFace
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_acc = 100 * correct / total
    scheduler.step()
    print(f"Epoch {epoch+1}: Loss {running_loss/len(train_loader):.4f}, Acc {train_acc:.2f}%")

torch.save(model.state_dict(), "arcface_model.pth")

False cpu
Количество людей: 31
Loaded


Epoch 1/2: 100%|██████████| 161/161 [02:18<00:00,  1.16it/s]


Epoch 1: Loss 32.0876, Acc 0.00%


Epoch 2/2: 100%|██████████| 161/161 [02:14<00:00,  1.20it/s]

Epoch 2: Loss 23.7918, Acc 0.00%


In [14]:
import torch.optim as optim
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FaceModel(num_classes=num_classes, embedding_dim=128).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
# Шедулер для уменьшения шага обучения
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

num_epochs = 1   # для начала хватит

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)   # классификационные логиты
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_acc = 100 * correct / total
    # Валидация
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    val_acc = 100 * val_correct / val_total
    
    scheduler.step()
    print(f"Train Loss: {running_loss/len(train_loader):.4f}, Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss/len(val_loader):.4f}, Acc: {val_acc:.2f}%")

# Сохраняем модель
torch.save(model.state_dict(), "face_model.pth")
print("Модель сохранена.")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Arsbul/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:13<00:00, 3.48MB/s]
Epoch 1/25: 100%|██████████| 129/129 [02:04<00:00,  1.03it/s]


Train Loss: 2.2132, Acc: 34.75% | Val Loss: 2.2253, Acc: 37.23%


Epoch 2/25:   3%|▎         | 4/129 [00:04<02:23,  1.15s/it]


KeyboardInterrupt: 

In [ ]:
import torch.nn.functional as F
from PIL import Image

# Загружаем обученную модель
model = FaceModel(num_classes=num_classes, embedding_dim=128)
model.load_state_dict(torch.load("face_model.pth", map_location=device))
model.to(device)
model.eval()

# Функция для получения эмбеддинга одного лица
def get_embedding(face_img_path):
    # Детектируем и выравниваем лицо
    img = Image.open(face_img_path).convert("RGB")
    face_tensor = mtcnn(img)   # [3, 160, 160] или None, если лица нет
    if face_tensor is None:
        raise ValueError("Лицо не найдено")
    # Применяем ту же нормализацию, что и при обучении
    transform = transforms.Compose([
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    face_tensor = transform(face_tensor).unsqueeze(0).to(device)  # [1, 3, 160, 160]
    with torch.no_grad():
        emb = model(face_tensor, return_embedding=True)
    return emb.squeeze().cpu()

# Сравниваем два изображения
emb1 = get_embedding("photo1.jpg")
emb2 = get_embedding("photo2.jpg")

# Нормализуем векторы для косинусного сходства
emb1_norm = F.normalize(emb1, dim=0)
emb2_norm = F.normalize(emb2, dim=0)
similarity = torch.dot(emb1_norm, emb2_norm).item()   # от -1 до 1
print(f"Сходство: {similarity:.4f}")
if similarity > 0.6:   # порог подбирается экспериментально
    print("Один и тот же человек")
else:
    print("Разные люди")

In [6]:
!pip install facenet-pytorch

ERROR: Could not install packages due to an OSError: [WinError 5] Отказано в доступе: 'C:\\Users\\Arsbul\\Desktop\\yolo8eyes_detection\\venv\\Lib\\site-packages\\~orch\\lib\\asmjit.dll'
Check the permissions.




  Attempting uninstall: torch
    Found existing installation: torch 2.7.1+cu118
    Uninstalling torch-2.7.1+cu118:
      Successfully uninstalled torch-2.7.1+cu118
